# Phase 1: Basic Concepts - Building Agent Infrastructure

## Overview
Now that you understand agent infrastructure concepts, let's implement core components.

In this phase, you'll build:
1. Agent identification and registration
2. Action logging and auditing systems
3. Attribution mechanisms
4. Basic permission systems
5. Simple monitoring

**What you'll learn:** How infrastructure actually works

---

## Part 1: Agent Identification System

Every agent needs a unique identity that infrastructure can track.

In [ ]:
import uuid
from datetime import datetime
from typing import Dict, List, Any
from dataclasses import dataclass, field
import json

@dataclass
class Agent:
    """Represents an AI agent with unique identity."""
    name: str
    agent_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    agent_type: str = "generic"
    owner_id: str = ""
    version: str = "1.0"
    created_at: datetime = field(default_factory=datetime.now)
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "agent_id": self.agent_id,
            "name": self.name,
            "type": self.agent_type,
            "owner": self.owner_id,
            "version": self.version,
            "created": self.created_at.isoformat()
        }

class AgentRegistry:
    """Infrastructure component: Agent identification and registration."""
    
    def __init__(self):
        self.agents: Dict[str, Agent] = {}
        self.owner_agents: Dict[str, List[str]] = {}  # user -> agent_ids
    
    def register_agent(self, name: str, agent_type: str, owner_id: str) -> Agent:
        """Register a new agent in the infrastructure."""
        agent = Agent(
            name=name,
            agent_type=agent_type,
            owner_id=owner_id
        )
        self.agents[agent.agent_id] = agent
        
        if owner_id not in self.owner_agents:
            self.owner_agents[owner_id] = []
        self.owner_agents[owner_id].append(agent.agent_id)
        
        print(f"✓ Registered agent: {agent.name} (ID: {agent.agent_id})")
        return agent
    
    def get_agent(self, agent_id: str) -> Agent:
        """Lookup an agent by ID."""
        return self.agents.get(agent_id)
    
    def list_user_agents(self, owner_id: str) -> List[Agent]:
        """Get all agents for a user."""
        agent_ids = self.owner_agents.get(owner_id, [])
        return [self.agents[aid] for aid in agent_ids]
    
    def verify_agent_exists(self, agent_id: str) -> bool:
        """Check if an agent is registered."""
        return agent_id in self.agents

# Test agent registration
registry = AgentRegistry()
agent1 = registry.register_agent("shopping_bot", "ecommerce", "user_alice")
agent2 = registry.register_agent("email_agent", "communication", "user_alice")
agent3 = registry.register_agent("analyst", "data", "user_bob")

print(f"\nAgents for user_alice: {[a.name for a in registry.list_user_agents('user_alice')]}")
print(f"Agent lookup - {agent1.agent_id}: {registry.get_agent(agent1.agent_id).name}")

## Part 2: Action Logging and Auditing

Infrastructure must log all agent actions for auditing and investigation.

In [ ]:
@dataclass
class ActionLog:
    """Record of an agent action for auditing."""
    agent_id: str
    owner_id: str
    action_type: str
    parameters: Dict[str, Any]
    result: str
    timestamp: datetime = field(default_factory=datetime.now)
    status: str = "success"
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "agent_id": self.agent_id,
            "owner_id": self.owner_id,
            "action": self.action_type,
            "parameters": self.parameters,
            "result": self.result,
            "timestamp": self.timestamp.isoformat(),
            "status": self.status
        }

class AuditSystem:
    """Infrastructure component: Action logging and auditing."""
    
    def __init__(self, registry: AgentRegistry):
        self.registry = registry
        self.logs: List[ActionLog] = []
    
    def log_action(self, agent_id: str, action_type: str, 
                   parameters: Dict[str, Any], result: str) -> ActionLog:
        """Log an agent action."""
        agent = self.registry.get_agent(agent_id)
        if not agent:
            raise ValueError(f"Unknown agent: {agent_id}")
        
        log = ActionLog(
            agent_id=agent_id,
            owner_id=agent.owner_id,
            action_type=action_type,
            parameters=parameters,
            result=result
        )
        self.logs.append(log)
        return log
    
    def get_agent_history(self, agent_id: str) -> List[ActionLog]:
        """Get all actions by an agent."""
        return [log for log in self.logs if log.agent_id == agent_id]
    
    def get_user_history(self, owner_id: str) -> List[ActionLog]:
        """Get all actions by agents owned by a user."""
        return [log for log in self.logs if log.owner_id == owner_id]
    
    def search_actions(self, agent_id: str = None, action_type: str = None,
                      start_time: datetime = None) -> List[ActionLog]:
        """Search audit logs."""
        results = self.logs
        
        if agent_id:
            results = [l for l in results if l.agent_id == agent_id]
        if action_type:
            results = [l for l in results if l.action_type == action_type]
        if start_time:
            results = [l for l in results if l.timestamp >= start_time]
        
        return results
    
    def print_audit_trail(self, logs: List[ActionLog]):
        """Print formatted audit trail."""
        for log in logs:
            print(f"[{log.timestamp.strftime('%H:%M:%S')}] "
                  f"Agent: {log.agent_id} | "
                  f"Action: {log.action_type} | "
                  f"Result: {log.result}")

# Test auditing
audit = AuditSystem(registry)

# Log some actions
audit.log_action(agent1.agent_id, "purchase", 
                {"item": "laptop", "price": 999}, "success")
audit.log_action(agent1.agent_id, "payment", 
                {"amount": 999}, "success")
audit.log_action(agent2.agent_id, "send_email", 
                {"to": "user@example.com", "subject": "Hello"}, "success")
audit.log_action(agent3.agent_id, "query_data", 
                {"table": "sales", "limit": 1000}, "success")

print("\n=== Agent 1 Audit Trail ===")
audit.print_audit_trail(audit.get_agent_history(agent1.agent_id))

print("\n=== User Alice's Activity ===")
audit.print_audit_trail(audit.get_user_history("user_alice"))

## Part 3: Permission System

Infrastructure controls what agents are allowed to do.

In [ ]:
class PermissionSystem:
    """Infrastructure component: Permission and capability management."""
    
    def __init__(self):
        self.agent_permissions: Dict[str, List[str]] = {}  # agent_id -> capabilities
        self.action_limits: Dict[str, Dict[str, int]] = {}  # agent_id -> limits
    
    def grant_permission(self, agent_id: str, capability: str):
        """Grant a capability to an agent."""
        if agent_id not in self.agent_permissions:
            self.agent_permissions[agent_id] = []
        
        if capability not in self.agent_permissions[agent_id]:
            self.agent_permissions[agent_id].append(capability)
            print(f"✓ Granted {capability} to {agent_id}")
    
    def revoke_permission(self, agent_id: str, capability: str):
        """Revoke a capability from an agent."""
        if agent_id in self.agent_permissions:
            self.agent_permissions[agent_id] = [
                c for c in self.agent_permissions[agent_id] if c != capability
            ]
            print(f"✓ Revoked {capability} from {agent_id}")
    
    def has_permission(self, agent_id: str, capability: str) -> bool:
        """Check if agent has a capability."""
        return capability in self.agent_permissions.get(agent_id, [])
    
    def set_limit(self, agent_id: str, action: str, limit: int):
        """Set action limit for an agent."""
        if agent_id not in self.action_limits:
            self.action_limits[agent_id] = {}
        self.action_limits[agent_id][action] = limit
        print(f"✓ Set limit: {action} <= {limit} for {agent_id}")
    
    def check_limit(self, agent_id: str, action: str, value: int) -> bool:
        """Check if action is within limits."""
        if agent_id not in self.action_limits:
            return True
        
        limit = self.action_limits[agent_id].get(action)
        if limit is None:
            return True
        
        return value <= limit
    
    def list_permissions(self, agent_id: str) -> List[str]:
        """List all permissions for an agent."""
        return self.agent_permissions.get(agent_id, [])

# Test permission system
permissions = PermissionSystem()

# Grant permissions
permissions.grant_permission(agent1.agent_id, "purchase")
permissions.grant_permission(agent1.agent_id, "make_payment")
permissions.grant_permission(agent2.agent_id, "send_email")
permissions.grant_permission(agent3.agent_id, "query_database")

# Set limits
permissions.set_limit(agent1.agent_id, "purchase_amount", 1000)
permissions.set_limit(agent1.agent_id, "daily_spending", 5000)
permissions.set_limit(agent2.agent_id, "emails_per_hour", 100)

# Check permissions
print(f"\n=== Permission Checks ===")
print(f"Can agent1 purchase? {permissions.has_permission(agent1.agent_id, 'purchase')}")
print(f"Can agent1 delete files? {permissions.has_permission(agent1.agent_id, 'delete_files')}")
print(f"Can agent2 send email? {permissions.has_permission(agent2.agent_id, 'send_email')}")

print(f"\n=== Limit Checks ===")
print(f"$500 purchase within limit? {permissions.check_limit(agent1.agent_id, 'purchase_amount', 500)}")
print(f"$1500 purchase within limit? {permissions.check_limit(agent1.agent_id, 'purchase_amount', 1500)}")
print(f"50 emails/hour OK? {permissions.check_limit(agent2.agent_id, 'emails_per_hour', 50)}")
print(f"150 emails/hour OK? {permissions.check_limit(agent2.agent_id, 'emails_per_hour', 150)}")

## Part 4: Basic Monitoring

Infrastructure monitors agent behavior for anomalies.

In [ ]:
class SimpleMonitor:
    """Infrastructure component: Basic agent monitoring."""
    
    def __init__(self):
        self.agent_stats: Dict[str, Dict[str, int]] = {}
        self.alerts: List[str] = []
    
    def record_action(self, agent_id: str, action_type: str):
        """Record an action for monitoring."""
        if agent_id not in self.agent_stats:
            self.agent_stats[agent_id] = {}
        
        if action_type not in self.agent_stats[agent_id]:
            self.agent_stats[agent_id][action_type] = 0
        
        self.agent_stats[agent_id][action_type] += 1
    
    def detect_unusual_activity(self, agent_id: str, 
                               normal_rate: Dict[str, int]) -> List[str]:
        """Detect if agent activity is unusual."""
        alerts = []
        
        if agent_id not in self.agent_stats:
            return alerts
        
        for action, current_count in self.agent_stats[agent_id].items():
            expected = normal_rate.get(action, 0)
            if current_count > expected * 2:  # More than 2x normal
                alerts.append(
                    f"ALERT: {agent_id} doing {action} at {current_count}x normal rate"
                )
        
        return alerts
    
    def get_agent_stats(self, agent_id: str) -> Dict[str, int]:
        """Get activity statistics for an agent."""
        return self.agent_stats.get(agent_id, {})
    
    def print_statistics(self):
        """Print monitoring statistics."""
        print("\n=== Agent Activity Statistics ===")
        for agent_id, stats in self.agent_stats.items():
            print(f"\n{agent_id}:")
            for action, count in stats.items():
                print(f"  {action}: {count}")

# Test monitoring
monitor = SimpleMonitor()

# Simulate normal activity
for _ in range(3):
    monitor.record_action(agent1.agent_id, "purchase")
    monitor.record_action(agent1.agent_id, "payment")

for _ in range(1):
    monitor.record_action(agent2.agent_id, "send_email")

# Simulate unusual activity
for _ in range(10):
    monitor.record_action(agent3.agent_id, "query_database")

monitor.print_statistics()

# Detect anomalies
print("\n=== Anomaly Detection ===")
normal_rates = {
    "purchase": 3,
    "payment": 3,
    "send_email": 1,
    "query_database": 2
}

for agent_id in monitor.agent_stats.keys():
    alerts = monitor.detect_unusual_activity(agent_id, normal_rates)
    if alerts:
        for alert in alerts:
            print(f"  {alert}")

## Summary: What You Built

In this phase, you created:

1. ✅ **Agent Registry** - Identify and track agents
2. ✅ **Audit System** - Log and audit all actions
3. ✅ **Permission System** - Control what agents can do
4. ✅ **Monitoring System** - Detect unusual behavior

These are the foundations of agent infrastructure.

### Key Insights

- **Attribution is foundational** - You can't control what you can't identify
- **Logging enables accountability** - Complete audit trails show what happened
- **Permissions are preventive** - Control actions before they happen
- **Monitoring is detective** - Catch problems after they happen
- **Infrastructure is external** - Works without modifying agents

---

## Next Step

Ready for advanced patterns? Move to **Phase 2: Algorithms**

You'll implement:
- Inter-agent communication protocols
- Capability delegation systems
- Advanced anomaly detection
- Rollback and reversal mechanisms
- Resource allocation algorithms